# Hyper-parameter search
Random search over hidden size, layers, dropout, lr, seq_len on the VALIDATION split only; the test split is touched once at the end. Prefer `src.evaluation.walk_forward` for the final check.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import yaml, numpy as np, pandas as pd
from src.data.preprocess import load_ohlcv
dc = yaml.safe_load(open('../configs/data.yaml')); dc['path'] = '../' + dc['path']
bars = load_ohlcv(dc['path'], dc['resample']); bars.tail()

In [ ]:
import random, torch
from src.data.dataloader import build_loaders
from src.models.factory import build_model
from src.training.trainer import Trainer
from src.utils.seed import set_seed
mc = yaml.safe_load(open('../configs/model.yaml')); tc = yaml.safe_load(open('../configs/training.yaml')); tc['epochs'] = 8
random.seed(0); res = []
for trial in range(10):
    d = {**dc, 'seq_len': random.choice([32, 64, 96])}; m_ = {**mc, 'hidden': random.choice([32, 64, 128]), 'layers': random.choice([1, 2]), 'dropout': random.choice([0.1, 0.2, 0.4])}; t_ = {**tc, 'lr': random.choice([3e-4, 1e-3, 3e-3])}
    set_seed(1); D = build_loaders(d, bars); h = Trainer(build_model(m_, D['n_features'], d['seq_len']), t_, torch.device('cpu')).fit(D['train'], D['val'])
    res.append(dict(seq_len=d['seq_len'], **{k: m_[k] for k in ('hidden', 'layers', 'dropout')}, lr=t_['lr'], val=min(v for _, v in h)))
pd.DataFrame(res).sort_values('val')